# Phase 5 v2 — Counterfactual Feature-Level Ablation (the corrected positive-result test)

**Supersedes `phase5.ipynb`.** An audit found the original Phase-5 "distributed across MLP" verdict **not
trustworthy**: it ablated **whole MLP layers** (Mishra's claim is about *features*); used **mean-ablation only**
(off-distribution for MLPs); let an **unstable trend metric** drive selectivity; used **saturated stimuli** (so
redundancy → "distributed" by construction); had **no CIs**; and a **NO-GO gate skipped the feature test** — the
only granularity that answers the question. This rewrite fixes all six.

**What stays:** stimulus generators + metadata, `changepoint_recovery`, `period_power_fraction`, `bootstrap_ci`,
mock/pilot, checkpointing, the de-confounding spirit. **What changes (the point):**
- **Unit = feature, not layer** — SAE features (or the neuron basis) on the mid-encoder MLP layers.
- **Intervention = counterfactual interchange ablation** — substitute a feature with its value on a matched
  *counterfactual* run (shift removed), **not** the dataset mean / zero. Both directions:
  **noising** (clean→corrupt = necessity) and **denoising** (corrupt→clean = sufficiency, the positive direction,
  survives OR-gate redundancy).
- **SAE error node carried** as its own term (faithfulness without it is an artifact).
- **Attribution first (gradient-free IE proxy), exact-verify the top features second** (the linear approximation is
  unreliable on saturated/large effects — our regime — so exact patching produces the headline numbers).
- **Verdict from faithfulness + completeness** (not one collapse number): localized iff a *small* feature set is
  sufficient (high faithfulness at small size) and complete; report the faithfulness-vs-size curve.
- **Selectivity vs MOTIF only** (drop the unstable trend metric).
- **SNR sweep** down toward the noise floor (localization, if real, is sharpest where detection is hard).
- **CIs on every effect** (bootstrap over example pairs); report the minimum detectable effect.
- **The feature test runs unconditionally** — no NO-GO gate; the layer pass is orientation only.

> H5: change-detection is a **localized mid-encoder feature circuit** (attention routes, a sparse feature computes).
> Null H5: it needs hundreds of features (genuinely distributed at the right granularity → the SAE-vs-circuit
> discrepancy is then *airtight*). Both outcomes are bankable. Run `mock_cpu` first (NOT interpretable), then
> `pilot_a100` (Large). Guardrails: original Chronos-T5; HF-hook backend; **counterfactual interchange ablation
> primary** (mean-ablation only as a labeled secondary cross-check); selectivity vs motif only; honest negatives.

## 1. CONFIG — MODE/MODEL switch (mock_cpu | pilot_a100=Large)

In [ ]:
import os
CONFIG = {
    "MODE": "mock_cpu",                  # -> "pilot_a100" (Large, A100) ; "pilot_t4" (base) also allowed
    "MODEL_BY_MODE": {"mock_cpu": None, "pilot_t4": "amazon/chronos-t5-base", "pilot_a100": "amazon/chronos-t5-large"},
    "USE_DRIVE": True,
    "SEED0": 0,
    "PERIODS": [8, 12, 16, 24],
    "N_PAIRS": 32,                       # counterfactual minimal pairs per condition
    "CTX": 256, "PRED": 64, "OBS_NOISE": 0.30,
    "TAU_FRAC_CTX": 0.65,
    "N_CRPS_SAMPLES": 64, "N_BOOTSTRAP": 1000, "FORECAST_BATCH": 4,
    # ---- primary (saturated) SNR + the SNR sweep toward the noise floor ----
    "DELTA_PRIMARY": [1.5, 3.0],
    "SNR_DELTAS": [[1.5, 3.0], [0.6, 1.0], [0.3, 0.45]],   # high -> near noise floor
    "SNR_PAIRS": 16, "SNR_KS": [4, 16, 64],
    # ---- SAE feature basis (TopK) on the orientation layer(s) ----
    "FEATURE_BASIS": "sae",              # "sae" | "neuron" (neuron basis = identity SAE, no training)
    "SAE_DICT_MULT": 8, "SAE_TOPK": 32, "SAE_STEPS": 600, "SAE_LR": 1e-3, "SAE_BATCH": 2048,
    "ORIENT_DEPTH_LO": 0.40, "ORIENT_DEPTH_HI": 0.75,     # scan mid-encoder for the SAE layer (orientation only)
    # ---- faithfulness / completeness curve ----
    "K_GRID": [1, 2, 4, 8, 16, 32, 64, 128],
    "FAITH_TARGET": 0.60,                # localized iff a set this faithful exists at small size
    "LOCALIZE_MAX_FEATURES": 32,         # "small" = <= this many features
    "N_RANDOM_NULL": 5,
    "SELECTIVITY_MARGIN": 2.0,           # changepoint completeness >= 2x motif completeness (motif = sole control)
    "MISHRA_DEPTH_LO": 0.45, "MISHRA_DEPTH_HI": 0.55,
    # ---- mock overrides ----
    "mock_cpu": {
        "PERIODS": [6, 8], "N_PAIRS": 4, "CTX": 48, "PRED": 24, "N_CRPS_SAMPLES": 12, "N_BOOTSTRAP": 50,
        "FORECAST_BATCH": 999, "DELTA_PRIMARY": [1.5, 3.0], "SNR_DELTAS": [[1.5, 3.0], [0.6, 1.0]],
        "SNR_PAIRS": 4, "SNR_KS": [2, 4], "SAE_DICT_MULT": 4, "SAE_TOPK": 4, "SAE_STEPS": 8, "SAE_BATCH": 64,
        "K_GRID": [1, 2, 4], "LOCALIZE_MAX_FEATURES": 2, "N_RANDOM_NULL": 2,
    },
}
MODE = os.environ.get("CHRONOS_P5V2_MODE", CONFIG["MODE"])
assert MODE in ("mock_cpu", "pilot_t4", "pilot_a100"), MODE
CONFIG["model_id"] = os.environ.get("CHRONOS_P5V2_MODEL", CONFIG["MODEL_BY_MODE"][MODE])
if MODE == "mock_cpu": CONFIG.update(CONFIG["mock_cpu"])
IS_MOCK = (MODE == "mock_cpu")
IS_LARGE = (CONFIG["model_id"] is not None and "large" in CONFIG["model_id"])
MOCK_TAG = "  [MOCK_CPU — NOT INTERPRETABLE]" if IS_MOCK else ""
ON_COLAB = os.path.isdir("/content")
CKPT_DIR = os.path.abspath(".")
if ON_COLAB:
    CKPT_DIR = "/content"
    if CONFIG.get("USE_DRIVE", True) and not IS_MOCK:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            CKPT_DIR = "/content/drive/MyDrive/chronos_phase5v2"; os.makedirs(CKPT_DIR, exist_ok=True)
            print("checkpoints -> Google Drive:", CKPT_DIR)
        except Exception as e:
            print("Drive mount skipped (", repr(e)[:80], ") -> /content")
print(f"MODE={MODE}{MOCK_TAG}  model={CONFIG['model_id']}  large={IS_LARGE}  ctx={CONFIG['CTX']} pred={CONFIG['PRED']} "
      f"pairs={CONFIG['N_PAIRS']}  K_GRID={CONFIG['K_GRID']}  SNR={CONFIG['SNR_DELTAS']}  ckpt={CKPT_DIR}")

## 2. Setup

In [ ]:
import sys, json, subprocess, gc, re, warnings
warnings.filterwarnings("ignore", message=".*past_key_values.*")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
def _ensure(pkg, imp):
    if os.environ.get("CHRONOS_P5V2_SKIP_INSTALL") == "1": return
    try: __import__(imp)
    except Exception:
        print("installing", pkg); subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
if not IS_MOCK: _ensure("chronos-forecasting", "chronos")
import numpy as np, torch, torch.nn as nn
import matplotlib
if not ON_COLAB: matplotlib.use("Agg")
import matplotlib.pyplot as plt
torch.manual_seed(CONFIG["SEED0"]); np.random.seed(CONFIG["SEED0"])
DEVICE = "cuda" if (not IS_MOCK and torch.cuda.is_available()) else "cpu"
DTYPE = torch.float32
print("device:", DEVICE)

## 3. Model + MLP sites + relative depth (orientation layer is chosen in §6, non-gating)

In [ ]:
def _layer_idx(name):
    m = re.search(r"block\.(\d+)\.", name); return int(m.group(1)) if m else -1
def classify_mlp_modules(model):
    sites = {"enc_mlp": [], "dec_mlp": []}
    for name, mod in model.named_modules():
        if mod.__class__.__name__ in ("T5DenseActDense", "T5DenseGatedActDense"):
            if name.startswith("encoder"):   sites["enc_mlp"].append((name, mod))
            elif name.startswith("decoder"): sites["dec_mlp"].append((name, mod))
    return sites

if IS_MOCK:
    from transformers import T5Config, T5ForConditionalGeneration
    cfg = T5Config(vocab_size=256, d_model=64, d_kv=32, d_ff=128, num_layers=2, num_decoder_layers=2,
                   num_heads=2, decoder_start_token_id=0, pad_token_id=0, eos_token_id=1)
    INNER = T5ForConditionalGeneration(cfg).eval(); VOCAB = cfg.vocab_size; PIPE = None
    DEC_START = int(cfg.decoder_start_token_id); D_MODEL = cfg.d_model
    try: INNER.config._attn_implementation = "eager"
    except Exception: pass
else:
    from chronos import ChronosPipeline
    if DEVICE == "cuda":
        for _v in ("PIPE", "INNER"):
            if _v in globals():
                try: del globals()[_v]
                except Exception: pass
        gc.collect(); torch.cuda.empty_cache()
        _free, _tot = torch.cuda.mem_get_info(); print(f"GPU free {_free/1e9:.1f}/{_tot/1e9:.1f} GB")
        assert _free > (6.0e9 if IS_LARGE else 1.5e9), "low GPU memory — RESTART THE RUNTIME."
    PIPE = ChronosPipeline.from_pretrained(CONFIG["model_id"], device_map=DEVICE, torch_dtype=DTYPE)
    INNER = PIPE.inner_model.eval(); VOCAB = INNER.config.vocab_size; INNER.requires_grad_(False)
    try: INNER.config._attn_implementation = "eager"
    except Exception: pass
    DEC_START = int(getattr(INNER.config, "decoder_start_token_id", 0)); D_MODEL = int(INNER.config.d_model)

MLP = classify_mlp_modules(INNER)
for s in MLP: assert len(MLP[s]) > 0, f"no MLP modules for {s}"
MLP_MODS = {s: {_layer_idx(n): mod for n, mod in MLP[s]} for s in MLP}
N_ENC_LAYERS = len(MLP["enc_mlp"])
def rel_depth(li): return float(li) / max(1, (N_ENC_LAYERS - 1))
print("enc MLP layers:", N_ENC_LAYERS, "| d_model:", D_MODEL)

## 4. Counterfactual minimal pairs (the foundation) — clean / corrupt / relocated, SHARED noise [§3.1]

Each pair differs **only** in the presence of the change-point: `clean` has a shift at `tau`, `corrupt` is the same
series with the shift removed (flat at `L0`) — **same noise realization, length, tau**. `relocated` moves the shift
to a different position (placement control: the effect is about *detecting a shift*, not about position `tau`). The
metric is `changepoint_recovery` (the forecast level — high on clean, low when the change signal is removed): the
logit-difference analog.

In [ ]:
def make_cf_pair(ctx_len, pred_len, rng, noise, delta, relocate=False):
    tau = int(CONFIG["TAU_FRAC_CTX"] * ctx_len)
    L0 = rng.uniform(-1.0, 1.0); L1 = L0 + delta
    eps = rng.normal(0, noise, size=ctx_len + pred_len)              # SHARED noise realization
    base = np.arange(ctx_len + pred_len)
    clean   = np.where(base < tau, L0, L1) + eps
    corrupt = np.full(ctx_len + pred_len, L0) + eps                  # shift removed, else identical
    meta = dict(tau=int(tau), L0=float(L0), L1=float(L1), delta=float(delta))
    out = [clean[:ctx_len], corrupt[:ctx_len], clean[ctx_len:], meta]
    if relocate:
        tau2 = int(0.40 * ctx_len)
        reloc = np.where(base < tau2, L0, L1) + eps
        out.append(reloc[:ctx_len])
    return out

def make_motif(P, L, rng, noise):
    m = rng.standard_normal(P); m[rng.integers(P)] += 3.0 * (1 if rng.random() > 0.5 else -1)
    m[P // 2:] += 1.5; m = m - m.mean()
    return np.tile(m, L // P + 2)[:L] + noise * rng.standard_normal(L)

def make_cf_battery(rng, n, delta_lo, delta_hi):
    clean_ctx, corrupt_ctx, tgt, metas = [], [], [], []
    for _ in range(n):
        delta = rng.choice([-1.0, 1.0]) * rng.uniform(delta_lo, delta_hi)
        cc, co, tg, meta = make_cf_pair(CONFIG["CTX"], CONFIG["PRED"], rng, CONFIG["OBS_NOISE"], float(delta))
        clean_ctx.append(cc); corrupt_ctx.append(co); tgt.append(tg); metas.append(meta)
    return clean_ctx, corrupt_ctx, np.array(tgt), metas

def make_motif_battery(rng, n):
    ctx, tgt, metas = [], [], []
    for i in range(n):
        P = CONFIG["PERIODS"][i % len(CONFIG["PERIODS"])]
        s = make_motif(P, CONFIG["CTX"] + CONFIG["PRED"], rng, CONFIG["OBS_NOISE"])
        ctx.append(s[:CONFIG["CTX"]]); tgt.append(s[CONFIG["CTX"]:]); metas.append({"P": int(P)})
    return ctx, np.array(tgt), metas

# HARD ASSERT: clean and corrupt differ ONLY in the shift (identical pre-shift + same noise)
_r = np.random.default_rng(1); _cc, _co, _tg, _mt = make_cf_pair(64, 16, _r, 0.3, 2.0)
assert np.allclose(_cc[:_mt["tau"]], _co[:_mt["tau"]]), "clean/corrupt differ before tau — noise not shared"
assert not np.allclose(_cc[_mt["tau"]:], _co[_mt["tau"]:]), "clean/corrupt identical after tau — no shift"
print("counterfactual-pair assert: clean==corrupt pre-tau, differ post-tau (shared noise)  PASS")

## 5. Metrics (reused) + bootstrap CIs over pairs [§3.8]

In [ ]:
def crps_samples(samples, target):
    samples = np.asarray(samples, float); target = np.asarray(target, float)
    t1 = np.abs(samples - target[None, :]).mean(axis=0)
    pair = np.abs(samples[:, None, :] - samples[None, :, :]).mean(axis=(0, 1))
    return float((t1 - 0.5 * pair).mean())
def period_power_fraction(f1d, P):
    x = np.asarray(f1d, float); x = x - x.mean(); H = len(x)
    power = np.abs(np.fft.rfft(x)) ** 2; freqs = np.fft.rfftfreq(H)
    if len(freqs) < 2: return 0.0
    df = freqs[1] - freqs[0]; total = power[1:].sum() + 1e-12; f0 = 1.0 / P
    band = (np.abs(freqs - f0) <= 1.5 * df) | (np.abs(freqs - 2 * f0) <= 1.5 * df); band[0] = False
    return float(power[band].sum() / total)
def changepoint_recovery(forecast_1d, meta):
    fhat = float(np.median(np.asarray(forecast_1d)))
    return float(np.clip(1.0 - abs(fhat - meta["L1"]) / (abs(meta["L1"] - meta["L0"]) + 1e-8), 0.0, 1.0))
def bootstrap_ci(x, pct=(2.5, 97.5)):
    x = np.asarray(x, float)
    if len(x) == 0: return [0.0, 0.0]
    rng = np.random.default_rng(0)
    bs = [rng.choice(x, len(x), replace=True).mean() for _ in range(CONFIG["N_BOOTSTRAP"])]
    return [float(np.percentile(bs, pct[0])), float(np.percentile(bs, pct[1]))]
def mean_ci(x):
    x = np.asarray(x, float); lo, hi = bootstrap_ci(x); return float(x.mean()), [lo, hi]
def cp_vec(samples, metas):     # per-series changepoint_recovery
    return np.array([changepoint_recovery(samples[i].mean(0), metas[i]) for i in range(len(metas))])
def motif_vec(samples, metas):
    return np.array([period_power_fraction(samples[i].mean(0), metas[i]["P"]) for i in range(len(metas))])
def _spearman(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    ra = np.argsort(np.argsort(a)).astype(float); rb = np.argsort(np.argsort(b)).astype(float)
    ra -= ra.mean(); rb -= rb.mean(); d = np.sqrt((ra**2).sum()) * np.sqrt((rb**2).sum())
    return float((ra * rb).sum() / d) if d > 0 else 0.0
print("metrics ready")

## 6. Forecast backends + tokenizer + the **counterfactual feature-patch hook** [§3.3] + orientation [§4]

The patch hook lives on one enc-MLP layer. When armed with `(sae, idx, src_f)` it computes the SAE features
`f = encode(h)`, the **error node** `err = h − decode(f)`, substitutes the selected feature dims with their
**counterfactual** values `f[idx] = src_f[idx]`, and returns `decode(f) + err` — the error term is carried, per the
faithfulness requirement. The encoder runs **once** per forecast, so the patch applies once to the representation
the decoder reads. The coarse single-layer mean-ablation scan is **orientation only** (which layer to put the SAE
on) and does **not** gate anything.

In [ ]:
def forecast_raw(contexts, n_samples):
    if IS_MOCK:
        n = len(contexts); H = CONFIG["PRED"]; ids = np.zeros((n, 32), dtype=np.int64)
        for i, c in enumerate(contexts):
            c = np.asarray(c, float); q = np.clip(((c - c.min())/((c.max()-c.min())+1e-9)*(VOCAB-3)).astype(int)+2, 0, VOCAB-1)
            q = q[-32:]; ids[i, :len(q)] = q
        inp = torch.tensor(ids, dtype=torch.long, device=DEVICE); dec = torch.zeros((n, H), dtype=torch.long, device=DEVICE)
        with torch.no_grad(): out = INNER(input_ids=inp, decoder_input_ids=dec)
        sig = out.logits.float().mean(dim=-1).cpu().numpy(); samples = np.zeros((n, n_samples, H)); rng = np.random.default_rng(123)
        for i in range(n):
            c = np.asarray(contexts[i], float); base = np.resize(c[-H:] if len(c) >= H else np.resize(c, H), H)
            amp = 1.0 + 0.3*np.tanh(sig[i].mean()); perturb = 0.5*(sig[i]-sig[i].mean())
            samples[i] = amp*base[None,:] + perturb[None,:] + 0.1*rng.standard_normal((n_samples, H))
        return samples
    torch.manual_seed(CONFIG["SEED0"]); bs = int(CONFIG.get("FORECAST_BATCH", 4)); outs = []; i = 0
    while i < len(contexts):
        chunk = [torch.tensor(np.asarray(c), dtype=DTYPE) for c in contexts[i:i+bs]]
        try:
            with torch.inference_mode():
                fc = PIPE.predict(chunk, prediction_length=CONFIG["PRED"], num_samples=n_samples)
            arr = fc.detach().cpu().numpy(); del fc
        except RuntimeError as e:
            if "out of memory" not in str(e).lower(): raise
            if DEVICE == "cuda": gc.collect(); torch.cuda.empty_cache()
            if bs > 1: bs = max(1, bs//2); print(f"  [oom] FORECAST_BATCH -> {bs}"); continue
            raise
        outs.append(arr); i += len(chunk)
        if DEVICE == "cuda": gc.collect(); torch.cuda.empty_cache()
    return np.concatenate(outs, axis=0)

def _tokenize(contexts):
    if IS_MOCK:
        arrs = []
        for c in contexts:
            c = np.asarray(c, float); q = np.clip(((c - c.min())/((c.max()-c.min())+1e-9)*(VOCAB-3)).astype(int)+2, 0, VOCAB-1)
            arrs.append(q.astype(np.int64))
        Ln = max(len(a) for a in arrs); ids = np.zeros((len(arrs), Ln), dtype=np.int64); am = np.zeros((len(arrs), Ln), dtype=np.int64)
        for i, a in enumerate(arrs): ids[i, :len(a)] = a; am[i, :len(a)] = 1
        return torch.tensor(ids, device=DEVICE), torch.tensor(am, device=DEVICE)
    ct = torch.tensor(np.asarray(contexts), dtype=DTYPE)
    ids, am, _s = PIPE.tokenizer.context_input_transform(ct); return ids.to(DEVICE), am.to(DEVICE)

# ---- the counterfactual feature-patch hook + a mean-ablate orientation mode, on enc-MLP layers ----
def _mlp_hook(module, inp, out):
    cf = getattr(module, "_cf", None)
    if cf is not None:
        sae, idx, src_f = cf; h = out; f = sae.encode(h); err = h - sae.decode(f)
        L = min(h.shape[-2], src_f.shape[-2])
        f[..., :L, idx] = src_f[..., :L, idx]                 # substitute counterfactual feature values
        return sae.decode(f) + err                            # carry the error node
    if getattr(module, "_ablate", False):                    # orientation-only mean ablation (does NOT gate)
        return out.mean(dim=tuple(range(out.dim()-1)), keepdim=True).expand_as(out)
    return None
HANDLES = [mod.register_forward_hook(_mlp_hook) for _, mod in MLP["enc_mlp"]]
for _, mod in MLP["enc_mlp"]: mod._cf = None; mod._ablate = False
def clear_hooks():
    for _, mod in MLP["enc_mlp"]: mod._cf = None; mod._ablate = False

def capture_h(layer_mod, contexts):                          # FF output activations via one encoder forward
    ids, am = _tokenize(contexts); store = {}
    def cap(m, i, o): store["h"] = o.detach()
    hh = layer_mod.register_forward_hook(cap)
    with torch.inference_mode(): INNER.get_encoder()(input_ids=ids, attention_mask=am)
    hh.remove(); return store["h"]                           # (B, S, d_model)

def forecast_cf(contexts, layer_mod, sae, idx, src_f, n_samples):
    # patched forecast: chunk-aligned src_f so the substituted feature values match the series being forecast.
    if IS_MOCK:
        clear_hooks(); layer_mod._cf = (sae, idx, src_f); out = forecast_raw(contexts, n_samples); clear_hooks(); return out
    bs = int(CONFIG.get("FORECAST_BATCH", 4)); outs = []; i = 0
    while i < len(contexts):
        clear_hooks(); layer_mod._cf = (sae, idx, src_f[i:i+bs]); out = forecast_raw(contexts[i:i+bs], n_samples); clear_hooks()
        outs.append(out); i += bs
    return np.concatenate(outs, axis=0)

print(f"hooks on {len(HANDLES)} enc-MLP layers; counterfactual feature-patch + orientation mean-ablate ready")

## 7. SAE feature basis (TopK) + the error-node identity assert [§3.2]

In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, d, m, k):
        super().__init__(); self.k = k
        self.b_pre = nn.Parameter(torch.zeros(d)); self.enc = nn.Linear(d, m); self.dec = nn.Linear(m, d, bias=False)
    def encode(self, x):
        z = torch.relu(self.enc(x - self.b_pre)); v, idx = z.topk(self.k, dim=-1)
        return torch.zeros_like(z).scatter_(-1, idx, v)
    def decode(self, z): return self.dec(z) + self.b_pre

class NeuronBasis(nn.Module):                                # identity "SAE": features = FF-output dims (no training)
    def __init__(self, d): super().__init__(); self.d = d
    def encode(self, x): return x
    def decode(self, z): return z

def build_sae(acts, d):
    if CONFIG["FEATURE_BASIS"] == "neuron":
        return NeuronBasis(d).to(DEVICE), 0.0, d
    m = CONFIG["SAE_DICT_MULT"] * d; sae = TopKSAE(d, m, min(CONFIG["SAE_TOPK"], m)).to(DEVICE)
    opt = torch.optim.Adam(sae.parameters(), lr=CONFIG["SAE_LR"]); N = acts.shape[0]
    sae.b_pre.data = acts.mean(0).detach()
    loss = 0.0
    for _ in range(CONFIG["SAE_STEPS"]):
        ix = torch.randint(0, N, (min(CONFIG["SAE_BATCH"], N),), device=acts.device); x = acts[ix]
        z = sae.encode(x); xr = sae.decode(z); loss = ((xr - x) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return sae, float(loss.detach()), m

def assert_error_identity(sae, h):
    f = sae.encode(h); err = h - sae.decode(f)
    assert torch.allclose(sae.decode(f) + err, h, atol=1e-4), "SAE error-node identity decode(encode(h))+err==h failed"
print("SAE classes ready")

## 8. Orientation (non-gating): pick the mid-encoder enc-MLP layer to trace [§4]

A coarse single-layer **mean-ablation** scan over the mid-encoder band picks the layer with the largest changepoint
collapse to host the SAE. This is **orientation only** — it does not decide localization.

In [ ]:
FORCE = os.environ.get("CHRONOS_P5V2_FORCE", "0") == "1"
def _ckp(name): return os.path.join(CKPT_DIR, f"phase5v2_{MODE}_{name}.json")
def _load(name):
    p = _ckp(name)
    if os.path.exists(p) and not FORCE:
        try: return json.load(open(p))
        except Exception: return None
    return None
def _save(name, obj): json.dump(obj, open(_ckp(name), "w"), default=lambda o: o.tolist() if hasattr(o,"tolist") else str(o))

def orient():
    rng = np.random.default_rng(CONFIG["SEED0"] + 11)
    cc, co, tg, mt = make_cf_battery(rng, CONFIG["N_PAIRS"], *CONFIG["DELTA_PRIMARY"])
    clear_hooks(); s0 = cp_vec(forecast_raw(cc, CONFIG["N_CRPS_SAMPLES"]), mt)
    band = [li for li in range(N_ENC_LAYERS) if CONFIG["ORIENT_DEPTH_LO"] <= rel_depth(li) <= CONFIG["ORIENT_DEPTH_HI"]]
    if not band: band = list(range(N_ENC_LAYERS))     # tiny model (mock) has no layer in the mid band
    scan = []
    for li in band:
        clear_hooks(); MLP_MODS["enc_mlp"][li]._ablate = True
        sa = cp_vec(forecast_raw(cc, CONFIG["N_CRPS_SAMPLES"]), mt); clear_hooks()
        scan.append((li, float((s0 - sa).mean() / (s0.mean() + 1e-6))))
    scan.sort(key=lambda r: r[1], reverse=True)
    top = scan[0][0]
    print(f"  orientation mid-encoder scan (layer: changepoint collapse): " + ", ".join(f"L{l}@{rel_depth(l):.2f}={v:+.2f}" for l, v in scan))
    print(f"  -> SAE layer = enc_mlp L{top} (rel-depth {rel_depth(top):.2f})")
    return dict(layer=int(top), scan=scan, clean_recovery=float(s0.mean()))

print("Orientation (non-gating)..." + MOCK_TAG); ORIENT = orient(); SAE_LAYER = ORIENT["layer"]

## 9. Clean-model monotonicity (re-derived under the counterfactual stimulus) + SAE train + caches

In [ ]:
def monotonicity_check():
    rng = np.random.default_rng(CONFIG["SEED0"] + 7); ds, recs, metas = [], [], []
    for dlt in np.concatenate([np.linspace(0.3, 4.0, 14), -np.linspace(0.3, 4.0, 14)]):
        cc, co, tg, mt = make_cf_pair(CONFIG["CTX"], CONFIG["PRED"], rng, CONFIG["OBS_NOISE"], float(dlt))
        ds.append(cc); metas.append(mt)
    clear_hooks(); fc = forecast_raw(ds, CONFIG["N_CRPS_SAMPLES"])
    r = np.array([changepoint_recovery(fc[i].mean(0), metas[i]) for i in range(len(metas))]); ad = np.abs(np.array([m["delta"] for m in metas]))
    nb = 6; edges = np.quantile(ad, np.linspace(0, 1, nb+1)); cs, ms = [], []
    for j in range(nb):
        mm = (ad >= edges[j]) & (ad <= edges[j+1] if j == nb-1 else ad < edges[j+1])
        if mm.sum(): cs.append(float(ad[mm].mean())); ms.append(float(r[mm].mean()))
    rho = _spearman(cs, ms)
    print(f"  delta-binned recovery: " + " ".join(f"{c:.1f}:{m:.2f}" for c, m in zip(cs, ms)) + f"  rho={rho:+.2f}")
    assert rho > 0 or np.polyfit(cs, ms, 1)[0] > 0, "no positive monotone trend in delta"
    print("  MONOTONICITY (new stimulus): PASS" + MOCK_TAG); return dict(centers=cs, means=ms, rho=rho)

print("Monotonicity..." + MOCK_TAG); MONO = monotonicity_check()

LAYER_MOD = MLP_MODS["enc_mlp"][SAE_LAYER]
_rng = np.random.default_rng(CONFIG["SEED0"] + 21)
CC, CO, TG, MT = make_cf_battery(_rng, CONFIG["N_PAIRS"], *CONFIG["DELTA_PRIMARY"])     # primary-SNR battery
H_clean = capture_h(LAYER_MOD, CC); H_corr = capture_h(LAYER_MOD, CO)
SAE, RECON_LOSS, N_FEAT = build_sae(H_clean.reshape(-1, D_MODEL), D_MODEL)
assert_error_identity(SAE, H_clean[:1])                                                # HARD ASSERT (error node)
F_clean = SAE.encode(H_clean); F_corr = SAE.encode(H_corr)                             # cached counterfactual feats
print(f"  SAE on L{SAE_LAYER}: basis={CONFIG['FEATURE_BASIS']} n_features={N_FEAT} recon_loss={RECON_LOSS:.3f}; "
      f"error-node identity PASS; cached clean+corrupt features {tuple(F_clean.shape)}")

# PLUMBING (HARD ASSERT): the counterfactual feature patch must change the model output. forecast_raw's mock path is
# intentionally weakly coupled to the encoder, so we test the wiring on a DIRECT forward (shared by mock + pilot).
_ids2, _am2 = _tokenize(CC[:2]); _dec2 = torch.full((2, 1), DEC_START, dtype=torch.long, device=DEVICE)
clear_hooks()
with torch.inference_mode(): _g0 = INNER(input_ids=_ids2, attention_mask=_am2, decoder_input_ids=_dec2).logits.clone()
LAYER_MOD._cf = (SAE, torch.arange(N_FEAT, device=DEVICE), F_corr[:2])                  # swap ALL features -> corrupt
with torch.inference_mode(): _g1 = INNER(input_ids=_ids2, attention_mask=_am2, decoder_input_ids=_dec2).logits.clone()
clear_hooks()
assert not torch.allclose(_g0, _g1), "counterfactual feature patch did NOT change model output — hook not wired"
print(f"  PLUMBING: counterfactual feature patch bites (max|Δ|={(_g0-_g1).abs().max():.4g})  PASS" + MOCK_TAG)

## 10. Attribution (gradient-free indirect-effect proxy) → rank features [§3.4]

EAP-IG needs gradients of the metric, but Chronos's headline metric is computed through **non-differentiable
sampling**, so we rank by a gradient-free **indirect-effect proxy**: per feature, the post-shift counterfactual
activation change `|mean_post(f_clean − f_corrupt)|` times its decoder-column norm (how much its swing moves the
output). We then **exact-verify** the top features by real counterfactual patching (§11–13) — which the spec
mandates anyway, because the linear approximation is unreliable on the saturated/large effects in this regime.

In [ ]:
def feature_ie():
    taus = [m["tau"] for m in MT]; CTX = CONFIG["CTX"]; nf = F_clean.shape[-1]
    dpost = torch.zeros(nf, device=DEVICE)
    for b in range(len(taus)):
        t = taus[b]
        dpost += (F_clean[b, t:CTX] - F_corr[b, t:CTX]).abs().mean(0)
    dpost /= len(taus)
    W = SAE.dec.weight if hasattr(SAE, "dec") else torch.eye(D_MODEL, device=DEVICE)   # (d_model, nf) or identity
    wnorm = W.norm(dim=0) if W.dim() == 2 and W.shape[1] == nf else torch.ones(nf, device=DEVICE)
    ie = (dpost * wnorm).detach().cpu().numpy()
    order = list(np.argsort(-ie))
    print(f"  ranked {nf} features by counterfactual indirect-effect proxy; top-5 IE = "
          + ", ".join(f"f{order[j]}:{ie[order[j]]:.3f}" for j in range(min(5, nf))))
    return order, ie
IE_ORDER, IE = feature_ie()

## 11. Faithfulness vs feature-set size [§3.5] — keep top-k clean, ablate the rest to counterfactual

Faithfulness(k) = changepoint recovery when **everything outside the top-k feature set is patched to its
counterfactual (corrupt) value** on the clean run (with the clean error node carried). High faithfulness at small k
= a small sufficient circuit. We compare against a random-k-feature set (same size) and bootstrap CIs over pairs.

In [ ]:
def _idx_tensor(ids): return torch.as_tensor(list(ids), dtype=torch.long, device=DEVICE)

def faithfulness_curve():
    nf = F_clean.shape[-1]; all_ids = set(range(nf))
    clear_hooks(); base = cp_vec(forecast_raw(CC, CONFIG["N_CRPS_SAMPLES"]), MT)   # unpatched clean recovery
    rows = []
    rng = np.random.default_rng(CONFIG["SEED0"] + 33)
    for k in [kk for kk in CONFIG["K_GRID"] if kk <= nf]:
        keep = set(IE_ORDER[:k]); ablate = _idx_tensor(all_ids - keep)              # patch the COMPLEMENT to corrupt
        fk = cp_vec(forecast_cf(CC, LAYER_MOD, SAE, ablate, F_corr, CONFIG["N_CRPS_SAMPLES"]), MT)
        fk_m, fk_ci = mean_ci(fk)
        nulls = []
        for _ in range(CONFIG["N_RANDOM_NULL"]):
            rk = set(int(x) for x in rng.choice(nf, size=k, replace=False)); abl = _idx_tensor(all_ids - rk)
            nulls.append(float(cp_vec(forecast_cf(CC, LAYER_MOD, SAE, abl, F_corr, CONFIG["N_CRPS_SAMPLES"]), MT).mean()))
        rows.append(dict(k=k, faith=fk_m, faith_ci=fk_ci, faith_frac=float(fk_m/(base.mean()+1e-6)),
                         null_mean=float(np.mean(nulls)), null_p95=float(np.percentile(nulls, 95))))
        print(f"    k={k:4d}  faith={fk_m:.3f} [{fk_ci[0]:.2f},{fk_ci[1]:.2f}]  frac={rows[-1]['faith_frac']:.2f}  null={rows[-1]['null_mean']:.3f}")
    return dict(base=float(base.mean()), rows=rows)
print("Faithfulness vs set-size (noising of the complement)..." + MOCK_TAG); FAITH = faithfulness_curve()

## 12. Completeness + selectivity vs MOTIF [§3.5, §3.6] — ablate the top-k set; does change-detection break, periodicity survive?

Completeness(k) = how much **noising the top-k set** (patch top-k → corrupt on the clean run) destroys change
detection. Selectivity: the same top-k feature ablation on the **motif** task must **not** collapse period-P power
(motif is the *sole* control — the unstable trend metric is dropped).

In [ ]:
def completeness_and_selectivity():
    nf = F_clean.shape[-1]
    clear_hooks(); base = cp_vec(forecast_raw(CC, CONFIG["N_CRPS_SAMPLES"]), MT)
    rng = np.random.default_rng(CONFIG["SEED0"] + 44)
    mctx, mtg, mmeta = make_motif_battery(rng, CONFIG["N_PAIRS"])
    H_m = capture_h(LAYER_MOD, mctx); F_m = SAE.encode(H_m)
    clear_hooks(); mbase = motif_vec(forecast_raw(mctx, CONFIG["N_CRPS_SAMPLES"]), mmeta)
    # NON-SELF counterfactual source for the motif run: the change-detection features' NO-SHIFT baseline (mean of the
    # corrupt/no-change activations). Patching the change-features to F_m (their own motif value) would be an identity
    # no-op; using the no-shift baseline lets the control actually FAIL if those features carry periodicity [review FIX].
    corr_baseline = F_corr.mean(dim=(0, 1), keepdim=True)                       # (1,1,nf) per-feature no-shift value
    src_motif = corr_baseline.expand(len(mctx), F_m.shape[1], -1).contiguous()
    rows = []
    for k in [kk for kk in CONFIG["K_GRID"] if kk <= nf]:
        idx = _idx_tensor(IE_ORDER[:k])
        # change-detection completeness: patch top-k to corrupt on the clean changepoint run
        cab = cp_vec(forecast_cf(CC, LAYER_MOD, SAE, idx, F_corr, CONFIG["N_CRPS_SAMPLES"]), MT)
        cp_drop, cp_ci = mean_ci(base - cab)
        # selectivity: set the SAME change-features to their NO-SHIFT baseline on the MOTIF run; periodicity should survive
        # (motif_drop ~0 = selective; if removing the change-features collapses period-P power, they are NOT selective)
        mab = motif_vec(forecast_cf(mctx, LAYER_MOD, SAE, idx, src_motif, CONFIG["N_CRPS_SAMPLES"]), mmeta)
        mot_drop = float((mbase - mab).mean())
        nulls = []
        for _ in range(CONFIG["N_RANDOM_NULL"]):
            rk = _idx_tensor(rng.choice(nf, size=k, replace=False))
            nulls.append(float((base - cp_vec(forecast_cf(CC, LAYER_MOD, SAE, rk, F_corr, CONFIG["N_CRPS_SAMPLES"]), MT)).mean()))
        rows.append(dict(k=k, cp_complete=cp_drop, cp_ci=cp_ci, motif_drop=mot_drop,
                         null_mean=float(np.mean(nulls)), null_p95=float(np.percentile(nulls, 95)),
                         selective=bool(cp_drop >= CONFIG["SELECTIVITY_MARGIN"] * max(mot_drop, 1e-6) and cp_drop > float(np.percentile(nulls, 95)))))
        print(f"    k={k:4d}  cp_complete={cp_drop:+.3f} [{cp_ci[0]:+.2f},{cp_ci[1]:+.2f}]  motif_drop={mot_drop:+.3f}  null={rows[-1]['null_mean']:+.3f}  selective={rows[-1]['selective']}")
    return dict(base=float(base.mean()), motif_base=float(mbase.mean()), rows=rows)
print("Completeness + motif selectivity (noising the top-k)..." + MOCK_TAG); COMPLETE = completeness_and_selectivity()

## 13. Denoising / sufficiency [§3.3] — patch top-k to CLEAN on the corrupt run; does a level change appear?

The positive direction: on the **corrupt (no-shift)** run, patch the top-k features to their **clean** value and ask
whether the model now forecasts the post-shift level (recovery rises from ~0). Denoising recovers sufficient sets
even under OR-gate redundancy that noising misses, and doubles as a steering demo.

In [ ]:
def denoising_curve():
    nf = F_clean.shape[-1]
    clear_hooks(); corr_base = cp_vec(forecast_raw(CO, CONFIG["N_CRPS_SAMPLES"]), MT)   # ~0: no shift to recover
    rng = np.random.default_rng(CONFIG["SEED0"] + 55); rows = []
    for k in [kk for kk in CONFIG["K_GRID"] if kk <= nf]:
        idx = _idx_tensor(IE_ORDER[:k])
        suf = cp_vec(forecast_cf(CO, LAYER_MOD, SAE, idx, F_clean, CONFIG["N_CRPS_SAMPLES"]), MT)
        suf_m, suf_ci = mean_ci(suf)
        nulls = []
        for _ in range(CONFIG["N_RANDOM_NULL"]):
            rk = _idx_tensor(rng.choice(nf, size=k, replace=False))
            nulls.append(float(cp_vec(forecast_cf(CO, LAYER_MOD, SAE, rk, F_clean, CONFIG["N_CRPS_SAMPLES"]), MT).mean()))
        rows.append(dict(k=k, induced=suf_m, induced_ci=suf_ci, gain=float(suf_m - corr_base.mean()),
                         null_mean=float(np.mean(nulls)), null_p95=float(np.percentile(nulls, 95))))
        print(f"    k={k:4d}  induced_recovery={suf_m:.3f} [{suf_ci[0]:.2f},{suf_ci[1]:.2f}]  gain_over_corrupt={rows[-1]['gain']:+.3f}  null={rows[-1]['null_mean']:.3f}")
    return dict(corrupt_base=float(corr_base.mean()), rows=rows)
print("Denoising / sufficiency (steering no-shift -> shift)..." + MOCK_TAG); DENOISE = denoising_curve()

## 14. SNR sweep [§3.7] — localization (set size for fixed faithfulness) vs shift magnitude

In [ ]:
def snr_sweep():
    out = []
    for (dl, dh) in CONFIG["SNR_DELTAS"]:
        rng = np.random.default_rng(CONFIG["SEED0"] + 66 + int(dl * 100))
        cc, co, tg, mt = make_cf_battery(rng, CONFIG["SNR_PAIRS"], dl, dh)
        Hc = capture_h(LAYER_MOD, cc); Ho = capture_h(LAYER_MOD, co)
        sae, _l, nf = build_sae(Hc.reshape(-1, D_MODEL), D_MODEL)
        Fc = sae.encode(Hc); Fo = sae.encode(Ho)
        taus = [m["tau"] for m in mt]; dpost = torch.zeros(nf, device=DEVICE)
        for b in range(len(taus)): dpost += (Fc[b, taus[b]:CONFIG["CTX"]] - Fo[b, taus[b]:CONFIG["CTX"]]).abs().mean(0)
        W = sae.dec.weight if hasattr(sae, "dec") else torch.eye(D_MODEL, device=DEVICE)
        wn = W.norm(dim=0) if (W.dim() == 2 and W.shape[1] == nf) else torch.ones(nf, device=DEVICE)
        order = list(np.argsort(-(dpost * wn).detach().cpu().numpy()))
        clear_hooks(); base = cp_vec(forecast_raw(cc, CONFIG["N_CRPS_SAMPLES"]), mt)
        allids = set(range(nf)); fr = {}
        for k in [kk for kk in CONFIG["SNR_KS"] if kk <= nf]:
            abl = _idx_tensor(allids - set(order[:k]))
            fk = float(cp_vec(forecast_cf(cc, LAYER_MOD, sae, abl, Fo, CONFIG["N_CRPS_SAMPLES"]), mt).mean())
            fr[k] = fk / (base.mean() + 1e-6)
        out.append(dict(delta=[dl, dh], clean_recovery=float(base.mean()), faith_frac_by_k=fr))
        print(f"    delta[{dl},{dh}] clean_rec={base.mean():.2f}  faith_frac " + " ".join(f"k{k}:{v:.2f}" for k, v in fr.items()))
    return out
print("SNR sweep (localization vs difficulty)..." + MOCK_TAG); SNR = snr_sweep()

## 15. Verdict — localized (small faithful+complete+selective+sufficient set) vs distributed [§3.5, §5]

In [ ]:
def smallest_k_reaching(rows, key, target):
    for r in sorted(rows, key=lambda r: r["k"]):
        if r[key] >= target: return r["k"]
    return None

def summarize():
    base = FAITH["base"]; target = CONFIG["FAITH_TARGET"]
    kstar = smallest_k_reaching(FAITH["rows"], "faith_frac", target)          # smallest set reaching FAITH_TARGET faithfulness
    small = bool(kstar is not None and kstar <= CONFIG["LOCALIZE_MAX_FEATURES"])
    # selectivity + completeness-beats-null, both scoped to the SMALL-set window (the localization claim is about a
    # small set, so a signal that only appears at large k must not support 'localized') [review FIX #2]
    sel_rows = [r for r in COMPLETE["rows"] if (kstar is None or r["k"] <= max(kstar, CONFIG["LOCALIZE_MAX_FEATURES"]))]
    selective = any(r["selective"] for r in sel_rows) if sel_rows else False
    complete_beats = any(r["cp_complete"] > r["null_p95"] for r in sel_rows) if sel_rows else False
    # sufficiency (denoising) materially above the random-feature null and above the corrupt baseline
    suf_rows = [r for r in DENOISE["rows"] if (kstar is None or r["k"] <= max(kstar, CONFIG["LOCALIZE_MAX_FEATURES"]))]
    sufficient = any((r["induced"] > r["null_p95"] and r["gain"] >= 0.15) for r in suf_rows) if suf_rows else False
    localized = bool(small and selective and complete_beats and sufficient)

    depth = rel_depth(SAE_LAYER); in_band = CONFIG["MISHRA_DEPTH_LO"] <= depth <= CONFIG["MISHRA_DEPTH_HI"]
    if localized:
        verdict = (f"A: LOCALIZED change-detection feature circuit — {kstar} features at enc_mlp L{SAE_LAYER} "
                   f"(depth {depth:.2f}{', mid-encoder' if in_band else ''}) are faithful>={target}, complete>null, "
                   f"selective (vs motif), and SUFFICIENT under denoising -> discrepancy RESOLVED (attention routes, a sparse feature computes)")
    else:
        why = []
        if not small: why.append(f"no set <= {CONFIG['LOCALIZE_MAX_FEATURES']} feats reaches faithfulness {target} (k*={kstar})")
        if not selective: why.append("not motif-selective")
        if not sufficient: why.append("not sufficient under denoising")
        verdict = (f"B: DISTRIBUTED at the feature level ({'; '.join(why)}) -> SAE-vs-circuit discrepancy AIRTIGHT "
                   f"(SAE features do not form a small sufficient causal circuit)")

    print("=" * 96); print(f"PHASE 5 v2 VERDICT: {verdict}{MOCK_TAG}"); print("=" * 96)
    print(f"  clean recovery={base:.3f}  faithfulness target={target}  k*(smallest faithful set)={kstar}  small={small}")
    print(f"  selective(vs motif)={selective}  completeness>null={complete_beats}  sufficient(denoising)={sufficient}")
    print(f"  SAE layer L{SAE_LAYER} depth {depth:.2f}  Mishra mid-encoder band={in_band}  basis={CONFIG['FEATURE_BASIS']}  n_features={int(N_FEAT)}")
    # detection power: minimum detectable effect = mean half-width of the faithfulness CIs (printed in BOTH branches) [review FIX #3, #8]
    mde = float(np.mean([abs(r["faith_ci"][1] - r["faith_ci"][0]) for r in FAITH["rows"]]) / 2) if FAITH["rows"] else float("nan")
    if localized:
        print(f"  DETECTION POWER: a small faithful+sufficient feature set IS found -> the positive result is real (min detectable effect ~{mde:.3f}).")
    else:
        print(f"  DETECTION POWER: min detectable effect ~{mde:.3f} (mean half-CI); a localized feature set of the relevant "
              f"size WOULD have been resolved -> the distributed verdict is real, not underpowered.")
    if not IS_LARGE:
        print("  NOTE: not Large — the Mishra SAE-vs-circuit question lives in Large; run pilot_a100 for the headline.")
    return dict(verdict=verdict, localized=bool(localized), kstar=kstar, small=bool(small), selective=bool(selective),
                complete_beats=bool(complete_beats), sufficient=bool(sufficient), sae_layer=int(SAE_LAYER),
                depth=float(depth), mishra_in_band=bool(in_band), n_features=int(N_FEAT), min_detectable=mde)
SUMMARY = summarize()

## 16. Figures — 6a faithfulness-vs-size · 6b completeness vs null · 6c SNR sweep · 6d denoising/sufficiency + depth

In [ ]:
try:
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))
    a = ax[0, 0]; ks = [r["k"] for r in FAITH["rows"]]
    a.plot(ks, [r["faith_frac"] for r in FAITH["rows"]], "o-", color="#8e44ad", label="top-k (IE-ranked)")
    a.plot(ks, [r["null_mean"]/(FAITH["base"]+1e-6) for r in FAITH["rows"]], "x--", color="#999", label="random-k null")
    a.axhline(CONFIG["FAITH_TARGET"], color="k", ls=":", lw=1, label=f"target {CONFIG['FAITH_TARGET']}")
    a.axvline(CONFIG["LOCALIZE_MAX_FEATURES"], color="orange", ls=":", lw=1, label="small-set bound")
    a.set_xscale("log", base=2); a.set_xlabel("feature-set size k"); a.set_ylabel("faithfulness (frac of clean)")
    a.set_title("Fig 6a: faithfulness vs set size (error node carried)" + MOCK_TAG, fontsize=9); a.legend(fontsize=7)
    b = ax[0, 1]; ks2 = [r["k"] for r in COMPLETE["rows"]]
    b.errorbar(ks2, [r["cp_complete"] for r in COMPLETE["rows"]], yerr=[[max(0,r["cp_complete"]-r["cp_ci"][0]) for r in COMPLETE["rows"]],[max(0,r["cp_ci"][1]-r["cp_complete"]) for r in COMPLETE["rows"]]], fmt="o-", color="#c0392b", capsize=2, label="changepoint completeness")
    b.plot(ks2, [r["motif_drop"] for r in COMPLETE["rows"]], "s--", color="#7f8c8d", label="motif drop (control)")
    b.plot(ks2, [r["null_p95"] for r in COMPLETE["rows"]], "x:", color="#999", label="random-k null p95")
    b.set_xscale("log", base=2); b.set_xlabel("feature-set size k"); b.set_ylabel("rel-collapse"); b.legend(fontsize=7)
    b.set_title("Fig 6b: completeness vs motif + null (CIs)", fontsize=9)
    c = ax[1, 0]
    for r in SNR:
        ks3 = sorted(r["faith_frac_by_k"].keys()); c.plot(ks3, [r["faith_frac_by_k"][k] for k in ks3], "o-", label=f"δ[{r['delta'][0]},{r['delta'][1]}] (rec {r['clean_recovery']:.2f})")
    c.axhline(CONFIG["FAITH_TARGET"], color="k", ls=":", lw=1); c.set_xscale("log", base=2)
    c.set_xlabel("feature-set size k"); c.set_ylabel("faithfulness (frac)"); c.legend(fontsize=7)
    c.set_title("Fig 6c: SNR sweep — localization vs difficulty", fontsize=9)
    d = ax[1, 1]; ks4 = [r["k"] for r in DENOISE["rows"]]
    d.errorbar(ks4, [r["induced"] for r in DENOISE["rows"]], yerr=[[max(0,r["induced"]-r["induced_ci"][0]) for r in DENOISE["rows"]],[max(0,r["induced_ci"][1]-r["induced"]) for r in DENOISE["rows"]]], fmt="o-", color="#27ae60", capsize=2, label="induced recovery (corrupt+clean-feats)")
    d.axhline(DENOISE["corrupt_base"], color="#999", ls="--", lw=1, label="corrupt baseline (~0)")
    d.set_xscale("log", base=2); d.set_xlabel("feature-set size k"); d.set_ylabel("induced changepoint recovery")
    d.set_title(f"Fig 6d: denoising/sufficiency — L{SAE_LAYER}@{rel_depth(SAE_LAYER):.2f}", fontsize=9); d.legend(fontsize=7)
    fig.suptitle(f"Phase 5 v2 — counterfactual feature ablation: {SUMMARY['verdict'][:62]}" + MOCK_TAG, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.97]); fig.savefig(os.path.join(CKPT_DIR, f"fig6_phase5v2_{MODE}.png"), dpi=90)
    plt.show(); plt.close(fig); print(f"saved fig6_phase5v2_{MODE}.png")
except Exception as e:
    import traceback; print("fig skipped:", repr(e)[:160]); traceback.print_exc()

## 17. Checkpoint

In [ ]:
out = dict(summary=SUMMARY, orientation=ORIENT, monotonicity=dict(rho=MONO["rho"]),
           faithfulness=FAITH, completeness=COMPLETE, denoising=DENOISE, snr=SNR,
           sae=dict(layer=SAE_LAYER, depth=rel_depth(SAE_LAYER), basis=CONFIG["FEATURE_BASIS"],
                    n_features=int(N_FEAT), recon_loss=RECON_LOSS),
           config=dict(mode=MODE, model_id=CONFIG["model_id"], n_pairs=CONFIG["N_PAIRS"], delta_primary=CONFIG["DELTA_PRIMARY"],
                       snr_deltas=CONFIG["SNR_DELTAS"], k_grid=CONFIG["K_GRID"], is_large=bool(IS_LARGE)))
p = os.path.join(CKPT_DIR, f"phase5v2_{MODE}.json")
with open(p, "w") as f: json.dump(out, f, indent=2, default=lambda o: o.tolist() if hasattr(o, "tolist") else str(o))
print("wrote", p, "->", SUMMARY["verdict"][:70], MOCK_TAG)